<a href="https://colab.research.google.com/github/denisejroth/pixels-frames-transformers/blob/main/part3_transformers/12_detection_segmentation_and_counting/notebooks/12_detection_segmentation_and_counting_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pixels, Frames & Transformers
## Module 12: Detection, Segmentation and Counting · Solutions

**A Self-Paced Course in Computational Image and Video Analysis**  
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every ✏️ exercise is filled in with one possible answer and a short comment. There is usually more than one good way to solve an exercise, and your numbers may differ a little from run to run, so read the comments as "what to look for" rather than exact results.

Sometimes the question isn't what an image shows overall but what is in it and where: how many
people are at a rally, whether police appear, how much of a cereal box is taken up by a cartoon
mascot. **Object detection** draws boxes around things, **segmentation** outlines them pixel by
pixel, and newer models can find almost anything you can describe in words. This notebook uses them
to measure *parts* of images, and to build a classifier whose reasoning you can read.

In this notebook you will:

- detect people and everyday objects in the protest photos with a standard detector, count people,
  and check the counts against your crowd codes
- find things the standard detector doesn't know (police officers, protest signs) with an
  open-vocabulary detector, and validate them against your police and signs codes
- outline cartoon characters on cereal boxes with SAM 2 and measure how much of the box they cover
- build a two-level classifier that predicts "protest" from the objects found in an image

**Data.** The protest and King's Day photos with your codes from module 04 (all four variables),
and the Open Food Facts cereal packaging.

**Models.** All Apache 2.0: RT-DETR (`PekingU/rtdetr_r50vd`, trained on the 80 COCO categories),
OWLv2 (`google/owlv2-base-patch16-ensemble`) for open-vocabulary detection, and SAM 2.1
(`facebook/sam2.1-hiera-tiny`) for segmentation.

**Runtime.** You need a GPU: *Runtime → Change runtime type → T4 GPU*. The notebook takes about
fifteen minutes on a T4.

## 0. Setup

In [ ]:
!pip install -q "transformers>=4.57"

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests
import torch
from transformers import (AutoImageProcessor, AutoModelForObjectDetection, AutoProcessor,
                          AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor)
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, cohen_kappa_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

HEADERS = {"User-Agent": "PixelsFramesTransformers/1.0 (https://github.com/denisejroth/pixels-frames-transformers) course-notebook"}
BASE = "https://raw.githubusercontent.com/denisejroth/pixels-frames-transformers/main/shared/"
for helper in ["course_data.py", "course_torch.py"]:
    if not Path(helper).exists():
        response = requests.get(BASE + helper, headers=HEADERS, timeout=30)
        response.raise_for_status()  # stop if the download failed, rather than saving an error page
        Path(helper).write_text(response.text)
import course_data
import course_torch
from course_torch import DEVICE, load_rgb

print("Running on:", DEVICE)
ON_GPU = DEVICE == "cuda"

VARIABLES = ["protest", "crowd", "police", "signs"]
train_table, test_table = course_data.protest_train_test()
my_codes = pd.read_csv("my_codes.csv")
test_table = test_table.drop(columns=["protest"]).merge(my_codes[["title", *VARIABLES]], on="title")
packaging = course_data.get_packaging_corpus()

def draw_boxes(path, detections, ax, title=""):
    """Show an image with its detections (a table with label, score and box columns)."""
    img = load_rgb(path, 800)
    ax.imshow(img)
    for _, d in detections.iterrows():
        x0, y0, x1, y1 = d["box"]
        ax.add_patch(mpatches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color="yellow", linewidth=1.5))
        ax.text(x0, y0, f"{d['label']} {d['score']:.2f}", color="black", fontsize=6,
                bbox=dict(facecolor="yellow", alpha=0.7, pad=1, edgecolor="none"))
    ax.set_title(title, fontsize=8)
    ax.axis("off")

## 1. Standard object detection, and counting people

A detector outputs a list of **boxes**, each with a **category** and a **confidence score**. RT-DETR
is a transformer-based detector trained on COCO, a benchmark of photos labeled with 80 everyday
categories: *person*, *car*, *umbrella*, *backpack*, *traffic light*, and so on. No *police officer*,
no *banner*, no *flag*. As with ImageNet in module 06, the categories were chosen for a benchmark,
not for your research question.

We run it on all training and test images (the training images are needed in section 5) and keep
every box with a score of at least 0.3, so we can try different thresholds later without running
the model again. Images are resized to at most 800 pixels, and the boxes are in those pixel
coordinates.

In [ ]:
DETECTOR = "PekingU/rtdetr_r50vd"
det_processor = AutoImageProcessor.from_pretrained(DETECTOR)
detector = AutoModelForObjectDetection.from_pretrained(DETECTOR).eval().to(DEVICE)

@torch.no_grad()
def detect(paths, min_score=0.3, batch_size=8):
    rows = []
    for start in range(0, len(paths), batch_size):
        batch = paths[start:start + batch_size]
        images = [load_rgb(p, 800) for p in batch]
        inputs = det_processor(images=images, return_tensors="pt").to(DEVICE)
        outputs = detector(**inputs)
        results = det_processor.post_process_object_detection(
            outputs, threshold=min_score, target_sizes=[(img.height, img.width) for img in images])
        for path, img, res in zip(batch, images, results):
            for score, label, box in zip(res["scores"].tolist(), res["labels"].tolist(), res["boxes"].tolist()):
                rows.append({"local_path": path, "label": detector.config.id2label[label], "score": score,
                             "box": box, "area_share": (box[2] - box[0]) * (box[3] - box[1]) / (img.width * img.height)})
    return pd.DataFrame(rows, columns=["local_path", "label", "score", "box", "area_share"])

all_paths = train_table["local_path"].tolist() + test_table["local_path"].tolist()
start = time.time()
detections = detect(all_paths)
print(f"{len(detections)} boxes in {len(all_paths)} images, {time.time() - start:.0f}s")
detections["label"].value_counts().head(12)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3.4))
for ax, path in zip(axes, test_table["local_path"].head(4)):
    d = detections[(detections["local_path"] == path) & (detections["score"] >= 0.5)]
    draw_boxes(path, d, ax, f"{(d['label'] == 'person').sum()} people")
plt.tight_layout()
plt.show()

**Counting people** is then a matter of counting boxes labeled *person* above a threshold. Your
codebook defines *crowd* as "roughly ten or more people visible", so the obvious rule is: crowd if
the detector counts ten or more. Because the threshold of ten comes from the codebook, not from
looking at the test results, we can score this rule on the test set without cheating.

In [ ]:
def count_people(paths, threshold=0.5):
    people = detections[(detections["label"] == "person") & (detections["score"] >= threshold)]
    return people.groupby("local_path").size().reindex(paths, fill_value=0).to_numpy()

test_table["people"] = count_people(test_table["local_path"].tolist())
rule = (test_table["people"] >= 10).astype(int)
print(f"crowd rule (10+ people): balanced accuracy {balanced_accuracy_score(test_table['crowd'], rule):.2f}, "
      f"kappa {cohen_kappa_score(test_table['crowd'], rule):.2f}")

test_table.boxplot(column="people", by="crowd", figsize=(4, 3))
plt.suptitle("")
plt.title("people detected, by your crowd code")
plt.show()

> **✏️ Exercise 1**
>
> The count depends on the confidence threshold. Recompute the counts with thresholds of 0.3, 0.5
> and 0.7 and, for each, the agreement (balanced accuracy) of the 10+ rule with your crowd codes.
> Then look at the images where the detector counts far fewer people than you would: what do they
> have in common?

In [ ]:
# ✅ Solution
for t in [0.3, 0.5, 0.7]:
    counts = count_people(test_table["local_path"].tolist(), threshold=t)
    ba = balanced_accuracy_score(test_table["crowd"], (counts >= 10).astype(int))
    print(f"threshold {t}: median count {np.median(counts):.0f}, balanced accuracy of the 10+ rule {ba:.2f}")

missed = test_table[(test_table["crowd"] == 1) & (test_table["people"] < 10)]
fig, axes = plt.subplots(1, min(4, max(1, len(missed))), figsize=(12, 3.4), squeeze=False)
for ax, path in zip(axes[0], missed["local_path"].head(4)):
    d = detections[(detections["local_path"] == path) & (detections["score"] >= 0.5)]
    draw_boxes(path, d, ax, f"{(d['label'] == 'person').sum()} people detected")
plt.tight_layout()
plt.show()

# Lower thresholds count more people (and more false ones); higher thresholds miss partly hidden
# people. The undercounts are usually dense or distant crowds: people overlap, heads are a few
# pixels tall, and the detector finds the closest twenty and misses the hundreds behind them.
# Detectors also have a fixed maximum number of boxes. That is why crowd size estimation uses
# different methods (density maps that estimate people per area), and why counts from a detector
# are best used as "at least this many visible", not as crowd sizes.

## 2. Open-vocabulary detection: finding what you can describe

OWLv2 combines a detector with a CLIP-style text encoder (module 09): instead of 80 fixed
categories, it scores every candidate box against **text queries** you write. That lets us look for
the things in your codebook that COCO doesn't have. Its scores are on a different scale from
RT-DETR's; thresholds around 0.1 to 0.3 are typical.

In [ ]:
OWL = "google/owlv2-base-patch16-ensemble"
owl_processor = AutoProcessor.from_pretrained(OWL)
owl = AutoModelForZeroShotObjectDetection.from_pretrained(OWL).eval().to(DEVICE)

@torch.no_grad()
def find(paths, queries, min_score=0.05, batch_size=4):
    """Boxes for each text query, with scores; keeps everything above min_score."""
    rows = []
    for start in range(0, len(paths), batch_size):
        batch = paths[start:start + batch_size]
        images = [load_rgb(p, 800) for p in batch]
        inputs = owl_processor(text=[queries] * len(images), images=images, return_tensors="pt").to(DEVICE)
        outputs = owl(**inputs)
        results = owl_processor.post_process_grounded_object_detection(
            outputs, threshold=min_score, target_sizes=[(img.height, img.width) for img in images],
            text_labels=[queries] * len(images))
        for path, img, res in zip(batch, images, results):
            for score, label, box in zip(res["scores"].tolist(), res["text_labels"], res["boxes"].tolist()):
                rows.append({"local_path": path, "label": label, "score": score, "box": box,
                             "area_share": (box[2] - box[0]) * (box[3] - box[1]) / (img.width * img.height)})
    return pd.DataFrame(rows, columns=["local_path", "label", "score", "box", "area_share"])

QUERIES = ["a police officer", "a protest sign", "a banner", "a flag"]
start = time.time()
found = find(all_paths, QUERIES)
print(f"{len(found)} boxes in {time.time() - start:.0f}s")

fig, axes = plt.subplots(1, 4, figsize=(12, 3.4))
for ax, path in zip(axes, test_table["local_path"].head(4)):
    draw_boxes(path, found[(found["local_path"] == path) & (found["score"] >= 0.2)], ax)
plt.tight_layout()
plt.show()

To turn boxes into image-level codes, take the highest score per query in each image and compare it
with a threshold. We use 0.2, a common starting point, fixed in advance.

In [ ]:
def max_scores(paths, table, labels):
    best = table.groupby(["local_path", "label"])["score"].max().unstack()
    return best.reindex(index=paths, columns=labels).fillna(0)

test_scores = max_scores(test_table["local_path"].tolist(), found, QUERIES)
THRESHOLD = 0.2
owl_codes = pd.DataFrame({
    "police": (test_scores["a police officer"] >= THRESHOLD).astype(int).to_numpy(),
    "signs": (test_scores[["a protest sign", "a banner", "a flag"]].max(axis=1) >= THRESHOLD).astype(int).to_numpy(),
})
for v in ["police", "signs"]:
    print(f"{v:7s} balanced accuracy {balanced_accuracy_score(test_table[v], owl_codes[v]):.2f}, "
          f"kappa {cohen_kappa_score(test_table[v], owl_codes[v]):.2f}, "
          f"share yes: you {test_table[v].mean():.2f}, OWLv2 {owl_codes[v].mean():.2f}")

> **✏️ Exercise 2**
>
> Both the threshold and the wording of a query matter. (a) For *police*, compute the balanced accuracy
> for thresholds from 0.05 to 0.5. (b) Run `find` on the test images with three other wordings for
> police, such as `"police"`, `"a police officer in uniform"` and `"a person wearing a police
> uniform"`, and compare their balanced accuracy at a threshold of 0.2.
>
> Picking the threshold or wording that does best *here* would be tuning on the test set, so treat
> this as a sensitivity check. How would you choose them properly?

In [ ]:
# ✅ Solution
for t in [0.05, 0.1, 0.2, 0.3, 0.5]:
    pred = (test_scores["a police officer"] >= t).astype(int)
    print(f"threshold {t:.2f}: balanced accuracy {balanced_accuracy_score(test_table['police'], pred):.2f}")

WORDINGS = ["police", "a police officer in uniform", "a person wearing a police uniform"]
alt = find(test_table["local_path"].tolist(), WORDINGS)
alt_scores = max_scores(test_table["local_path"].tolist(), alt, WORDINGS)
for w in WORDINGS:
    pred = (alt_scores[w] >= 0.2).astype(int)
    print(f"{w:35s} balanced accuracy {balanced_accuracy_score(test_table['police'], pred):.2f}")

# Results usually move by several points across thresholds and wordings. To choose properly, hand-
# code a separate validation sample (say 100 images not in the test set), pick the threshold and
# wording there, and only then score the test set. Report the sensitivity either way.

## 3. Segmentation: how much of the box is the mascot?

A box is a rough outline: for a character with outstretched arms it includes a lot of background.
**SAM 2** (Segment Anything 2) outlines an object precisely when you point at it with a click or a
box. It doesn't know *what* it is outlining, which is why we combine it with OWLv2: OWLv2 finds the
cartoon character, SAM 2 traces it.

The measure we want is the **share of the package front covered by a character**, a plausible
indicator of child-oriented marketing.

In [ ]:
SAM = "facebook/sam2.1-hiera-tiny"
sam_processor = Sam2Processor.from_pretrained(SAM)
sam = Sam2Model.from_pretrained(SAM).eval().to(DEVICE)

@torch.no_grad()
def segment(path, boxes):
    """One mask per box, as boolean arrays the size of the image (at most 800 pixels)."""
    img = load_rgb(path, 800)
    inputs = sam_processor(images=img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors="pt").to(DEVICE)
    outputs = sam(**inputs, multimask_output=False)
    masks = sam_processor.post_process_masks(outputs.pred_masks.cpu(), inputs["original_sizes"].cpu())[0]
    return masks[:, 0].numpy().astype(bool)

N_PACK = len(packaging) if ON_GPU else 30
pack = packaging.head(N_PACK).reset_index(drop=True)
characters = find(pack["local_path"].tolist(), ["a cartoon character", "a mascot"], min_score=0.2)
print(f"{characters['local_path'].nunique()} of {len(pack)} packages have a character box with a score of 0.2 or more")

rows = []
for path, group in characters.groupby("local_path"):
    masks = segment(path, group["box"].tolist())
    union = masks.any(axis=0)
    rows.append({"local_path": path, "character_share": union.mean(),
                 "box_share": min(1.0, group["area_share"].sum())})
coverage = pack[["local_path", "nutriscore_grade"]].merge(pd.DataFrame(rows, columns=["local_path", "character_share", "box_share"]),
                                                          on="local_path", how="left").fillna({"character_share": 0, "box_share": 0})
coverage.describe().round(3)

In [ ]:
shown = coverage.sort_values("character_share", ascending=False).head(4)
fig, axes = plt.subplots(1, 4, figsize=(12, 3.6))
for ax, (_, row) in zip(axes, shown.iterrows()):
    img = load_rgb(row["local_path"], 800)
    boxes = characters[characters["local_path"] == row["local_path"]]["box"].tolist()
    mask = segment(row["local_path"], boxes).any(axis=0)
    ax.imshow(img)
    ax.imshow(np.ma.masked_where(~mask, mask), cmap="autumn", alpha=0.5)
    ax.set_title(f"character covers {row['character_share']:.0%}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

> **✏️ Exercise 3**
>
> (a) Compare the mask-based share with the box-based share (`box_share`). How much do boxes
> overstate the area, on average? (b) Do less healthy cereals give more of the box to characters?
> Compare the character share across Nutri-Score grades (a to e) with a box plot and a Spearman
> correlation. What else would you need to check before claiming that mascots are used to sell less
> healthy products?

In [ ]:
# ✅ Solution
has_char = coverage[coverage["character_share"] > 0]
print(f"packages with a character: {len(has_char)}; box share / mask share, median: "
      f"{(has_char['box_share'] / has_char['character_share'].clip(lower=1e-6)).median():.1f}")

graded = coverage[coverage["nutriscore_grade"].isin(list("abcde"))].copy()
graded["grade_number"] = graded["nutriscore_grade"].map({g: i for i, g in enumerate("abcde")})
rho, p = spearmanr(graded["character_share"], graded["grade_number"])
print(f"Spearman correlation of character share with Nutri-Score (a = 0 ... e = 4): {rho:.2f} (p = {p:.3f})")
graded.boxplot(column="character_share", by="nutriscore_grade", figsize=(5, 3.2))
plt.suptitle(""); plt.title("share of the front covered by characters"); plt.show()

# Boxes usually overstate the area by a factor of one and a half to three for characters, more for
# thin or irregular shapes. A positive correlation would fit the idea that characters are used for
# less healthy (often children's) cereals, but before claiming it you would: validate the detector
# (how many characters does it miss, how many logos or photos of fruit does it call characters?)
# against hand codes; check that the pattern isn't driven by a few brands with many variants; and
# consider confounders such as the target age group, which drives both design and recipe.

## 4. A two-level classifier: from objects to scenes

Scholz and colleagues (2025) propose classifying complex scenes such as protests in **two levels**:
first detect interpretable parts (people, signs, police, flags), then predict the scene from those
parts with a simple model. The second level is transparent: you can read which objects push an
image toward "protest". The price is that anything not captured by the detected parts is lost.

Our first level is everything we detected: the number of people and other COCO objects (RT-DETR),
and the best score per query (OWLv2). The second level is a logistic regression, trained on the
category labels of the training images and scored on your codes.

In [ ]:
COCO_KEEP = ["person", "car", "bicycle", "umbrella", "backpack", "handbag", "bus", "truck", "boat", "traffic light"]

def object_features(paths):
    d = detections[(detections["score"] >= 0.5) & detections["label"].isin(COCO_KEEP)]
    counts = d.groupby(["local_path", "label"]).size().unstack().reindex(index=paths, columns=COCO_KEEP).fillna(0)
    counts = np.log1p(counts).add_prefix("log count: ")
    owl_part = max_scores(paths, found, QUERIES).add_prefix("max score: ")
    return pd.concat([counts, owl_part], axis=1)

X_train = object_features(train_table["local_path"].tolist())
X_test = object_features(test_table["local_path"].tolist())
two_level = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000, C=0.5))
two_level.fit(X_train, train_table["category_label"])
print(course_data.log_score("two-level: detected objects -> logistic regression", test_table["protest"],
                            two_level.predict(X_test), module="12"))

coefficients = pd.Series(two_level[-1].coef_[0], index=X_train.columns).sort_values()
coefficients.plot.barh(figsize=(5, 4), title="what pushes an image toward 'protest'")
plt.axvline(0, color="gray")
plt.show()

> **✏️ Exercise 4**
>
> How much does interpretability cost? Add the CLIP embeddings of the images (`course_torch.clip_images`)
> to the object features, fit the same kind of model, and log it as `"objects + CLIP embeddings"`.
> Compare it with the two-level model and with the CLIP probe from module 09 on your scoreboard. When
> would you accept a lower score for a model you can read?

In [ ]:
# ✅ Solution
clip_train = course_torch.clip_images(train_table["local_path"])
clip_test = course_torch.clip_images(test_table["local_path"])
combined = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=3000, C=0.05))
combined.fit(np.hstack([X_train.to_numpy(), clip_train]), train_table["category_label"])
print(course_data.log_score("objects + CLIP embeddings", test_table["protest"],
                            combined.predict(np.hstack([X_test.to_numpy(), clip_test])), module="12"))
display(course_data.show_scores().head(10))

# Adding the embeddings usually helps, because they capture the scene as a whole (setting, mood,
# composition) that a list of objects misses, but the model becomes a black box again. A readable
# model is worth a few points when the explanation is part of the finding (which visual elements
# make an image a protest image?), when you need to show reviewers or courts what drives a
# classification, or when you want to check a model for shortcuts: a large weight on "umbrella"
# would tell you something about your training data's weather, not about protest.

## 5. A note on licenses

Detection tutorials often use the YOLO models from the Ultralytics library. Its recent versions are
licensed under **AGPL-3.0**, which requires that software built with it is released under the same
license, also when it is offered as a web service, unless you buy a commercial license. For a
research script that you publish anyway this may be fine, but it matters for tools you build for a
partner organization or plan to commercialize. The models in this notebook are Apache 2.0, which
only asks for attribution and a copy of the license. Always check the license of both the code and
the weights; they can differ.

> **✏️ Exercise 5**
>
> A newspaper reports "about 5,000 people" at a demonstration, and a reader sends you a photo of it in
> which a detector counts 140 people. Write three or four sentences on what the detector count can and
> cannot tell you about the size of the demonstration.

**✅ One possible answer**

The count is a lower bound on the number of people *visible and detectable* in one photo: it misses
people hidden behind others, people too small to detect, and everyone outside the frame or present
at another moment. A single photo can't support a claim about the total size of an event, which
depends on the area covered, density across that area and turnover over time. Crowd scientists
estimate size from area times density, often with several photos or aerial images, and report a
range. What the detector *can* do is give consistent, comparable counts across many photos taken
the same way, for example to compare the visible density of different protests, as long as those
counts are validated against hand counts on a sample.

In [ ]:
course_data.keep_my_files()